# Accelerated FinABSA Sequence Sequence Generation (Zero-Shot)

We have completely skipped training! Instead, we load the pristine **770-Million parameter** T5-Large mode (`amphora/FinABSA`) perfectly Zero-Shot. 

Because downloading a 3 Gigabyte model natively takes massive amounts of time on CPU connections, we deployed custom **Rust-based multi-threading network accelerators** (`hf_transfer`) instantly cutting the model fetch delay down to seconds.

In [ ]:
# Required Installations including Rust Accelerators
%pip install transformers datasets accelerate torch pandas numpy scikit-learn ipywidgets sentencepiece huggingface_hub hf_transfer

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 8.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## 1. High Velocity T5 Sequence Loading

In [3]:
import os
import torch
import json
import pandas as pd
from pathlib import Path

# Enable insane Rust-backed multi-threaded networking explicitly
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import classification_report, confusion_matrix

MODEL_NAME = "amphora/FinABSA"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Targeting processor: {device}")

print("Fetching 3-Gigabyte network via Multi-Thread Rust Transfer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()
model.to(device)
print("Colossal SEntFiN mathematically anchored in memory!")

Targeting processor: cpu
Fetching 3-Gigabyte network via Multi-Thread Rust Transfer...


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Colossal SEntFiN mathematically anchored in memory!


## 2. Dynamic Input Entity Target Locking
`amphora/FinABSA` expects us to swap the financial entity targeted securely with an explicit `[TGT]` token.

In [4]:
def parse_decisions(cell: str) -> dict:
    if not isinstance(cell, str) or not cell.strip(): return {}
    s = cell.strip()
    try: return json.loads(s)
    except: return json.loads(s.replace("'", '"'))

def load_target_samples(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    rows = []
    for _, r in df.iterrows():
        title = str(r["Title"])
        decisions = parse_decisions(str(r["Decisions"]))
        for aspect, sent in decisions.items():
            formatted_title = title.replace(aspect, "[TGT]")
            if "[TGT]" not in formatted_title:
                formatted_title = f"[TGT] {title}"
            
            # Amphora natively expects ABSA prefixes for certainty
            formatted_title = f"Sentiment Analysis: {formatted_title}"
            rows.append({"title": formatted_title, "label": sent.strip().lower()})
    return pd.DataFrame(rows)

CSV_PATH = "SEntFiN-v1.1.csv"
if not Path(CSV_PATH).is_file():
    CSV_PATH = r"C:\Users\prakh\Downloads\nlp\SEntFiN-v1.1.csv"

# CPU Limiter: 100 instances to prevent tensor looping freeze on your machine.
dataset_df = load_target_samples(CSV_PATH).sample(n=100, random_state=42).reset_index(drop=True)
print(f"Harvested {len(dataset_df)} instances for intensive generation eval:\n")
display(dataset_df.head(3))

Harvested 100 instances for intensive generation eval:



,title,label
0,"Sentiment Analysis: Bank, [TGT] to underperfor...",negative
1,Sentiment Analysis: [TGT] seeks licence to ent...,neutral
2,Sentiment Analysis: [TGT] recommends dividend ...,neutral


## 3. Generative String Extraction

In [2]:
preds = []
labels = dataset_df["label"].tolist()

print("Evaluating generating strings...")
with torch.no_grad():
    for i, row in dataset_df.iterrows():
        inputs = tokenizer(row["title"], return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        outputs = model.generate(**inputs, max_length=15)
        # The model hallucinates syntax literally e.g: `isNEUTRAL .`
        raw_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()
        
        # Smart extractor pulls exactly what the model attempted to map
        extracted_val = next((v for v in ["positive", "negative", "neutral"] if v in raw_text), "neutral")
        preds.append(extracted_val)

print("\n=== amphora/FinABSA Generative Evaluation ===\n")
valid_labels = ["positive", "negative", "neutral"]
print(classification_report(labels, preds, target_names=valid_labels))
display(pd.DataFrame(confusion_matrix(labels, preds, labels=valid_labels), index=valid_labels, columns=valid_labels))

NameError: name 'dataset_df' is not defined

## 4. UI Inference Graphic Vault

In [1]:
import ipywidgets as widgets
from IPython.display import display

def fetch_sentiment_from_t5(headline, entity):
   dynamic_format = headline.replace(entity, "[TGT]")
   if "[TGT]" not in dynamic_format:
      dynamic_format = f"[TGT] {headline}"
   
   enc = tokenizer(f"Sentiment Analysis: {dynamic_format}", return_tensors='pt')
   enc = {k: v.to(device) for k, v in enc.items()}
   with torch.no_grad(): out = model.generate(**enc, max_length=15)
   raw_out = tokenizer.decode(out[0], skip_special_tokens=True).lower()
   return next((v for v in ["positive", "negative", "neutral"] if v in raw_out), "UNKNOWN")

print("--- Swift Zero-Shot Inference Engine ---")
h_in = widgets.Text(placeholder='Headline...', layout=widgets.Layout(width='80%'))
a_in = widgets.Text(placeholder='Aspect...', layout=widgets.Layout(width='50%'))
btn = widgets.Button(description='Analyze Zero-Shot', button_style='primary')
ui_out = widgets.Output()

def test_seq(b):
    with ui_out:
        ui_out.clear_output()
        if h_in.value and a_in.value:
            print(f">>> Aspect: {a_in.value} | Model Decided: {fetch_sentiment_from_t5(h_in.value, a_in.value).upper()}")
btn.on_click(test_seq)
display(h_in, a_in, btn, ui_out)

--- Swift Zero-Shot Inference Engine ---


Text(value='', layout=Layout(width='80%'), placeholder='Headline...')

Text(value='', layout=Layout(width='50%'), placeholder='Aspect...')

Button(button_style='primary', description='Analyze Zero-Shot', style=ButtonStyle())

Output()

In [3]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

# ==========================================
# Phase 0: Mount the LLM Architecture
# ==========================================
print("--- Phase 0: Initializing FinABSA ---")
MODEL_NAME = "amphora/FinABSA"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()
model.to(device)

# ==========================================
# Phase 1: Load the Official Holdout Data
# ==========================================
print("\n--- Phase 1: Loading Official Test Set ---")
test_df = pd.read_csv("official_test_set.csv")
print(f"Columns found in your CSV: {test_df.columns.tolist()}")

# Auto-Detect the Column Names
TEXT_COL = "title" if "title" in test_df.columns else "headline"
ASPECT_COL = "aspect" if "aspect" in test_df.columns else "entity"

if "label" in test_df.columns:
    LABEL_COL = "label"
elif "sentiment_encoded" in test_df.columns:
    LABEL_COL = "sentiment_encoded"
elif "sentiment" in test_df.columns:
    LABEL_COL = "sentiment"
else:
    raise ValueError("CRITICAL ERROR: Could not find your sentiment column. Check the printed columns above!")

# Extract and map the true labels
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}

if test_df[LABEL_COL].dtype == object or test_df[LABEL_COL].dtype == str:
    labels = [str(x).lower() for x in test_df[LABEL_COL].tolist()]
else:
    labels = [ID2LABEL[int(x)] for x in test_df[LABEL_COL].tolist()]

preds = []

# ==========================================
# Phase 2: Zero-Shot Generation Engine
# ==========================================
print("\n--- Phase 2: Commencing Zero-Shot Generation ---")
print("Grab a coffee, this will take some time...")

with torch.no_grad():
    for i, row in tqdm(test_df.iterrows(), total=len(test_df), desc="T5 Generating Text"):
        
        # Format the text dynamically
        title = str(row[TEXT_COL])
        aspect = str(row[ASPECT_COL])
        
        formatted_title = title.replace(aspect, "[TGT]")
        if "[TGT]" not in formatted_title:
            formatted_title = f"[TGT] {title}"
            
        prompt = f"Sentiment Analysis: {formatted_title}"
        
        # Tokenize
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Generate prediction
        outputs = model.generate(**inputs, max_length=15)
        raw_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()
        
        # Extract signal
        extracted_val = next((v for v in ["positive", "negative", "neutral"] if v in raw_text), "neutral")
        preds.append(extracted_val)

# ==========================================
# Phase 3: Head-to-Head Output
# ==========================================
print("\n=== amphora/FinABSA Zero-Shot Evaluation (Official Test Set) ===\n")
valid_labels = ["positive", "negative", "neutral"]
print(classification_report(labels, preds, labels=valid_labels, target_names=valid_labels))

print("\nConfusion Matrix:")
display(pd.DataFrame(confusion_matrix(labels, preds, labels=valid_labels), index=valid_labels, columns=valid_labels))

--- Phase 0: Initializing FinABSA ---
Using compute device: cpu


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]


--- Phase 1: Loading Official Test Set ---
Columns found in your CSV: ['headline', 'aspect', 'sentiment', 'sentiment_encoded']

--- Phase 2: Commencing Zero-Shot Generation ---
Grab a coffee, this will take some time...


T5 Generating Text: 100%|██████████| 2882/2882 [1:18:19<00:00,  1.63s/it]



=== amphora/FinABSA Zero-Shot Evaluation (Official Test Set) ===

              precision    recall  f1-score   support

    positive       0.96      0.97      0.97      1015
    negative       0.96      0.94      0.95       764
     neutral       0.95      0.96      0.95      1103

    accuracy                           0.96      2882
   macro avg       0.96      0.95      0.96      2882
weighted avg       0.96      0.96      0.96      2882


Confusion Matrix:


,positive,negative,neutral
positive,984,13,18
negative,11,717,36
neutral,29,19,1055
